# NephroLLM — Execution Plan

1. Environment setup
2. Load Llama 3.2 3B Instruct
3. Define & Verify the base-model Inference
4. Load and validate the nephrology benchmark
5. Run a fresh baseline evaluation
6. Analyze baseline failures
7. Prepare fine-tuning dataset
8. Fine-tune using QLoRA
9. Save the LoRA adapter
10. Run the exact same evaluation on the fine-tuned model
11. Evaluate on held-out questions
12. Compare base vs fine-tuned model
13. Document findings, limitations, and conclusions

# NephroLLM — Fine-Tuning Llama 3.2 3B for Nephrology Q&A

Educational fine-tuning experiment using QLoRA to adapt Llama 3.2 3B Instruct to nephrology-focused questions.

**Objective:** Compare the base model against the QLoRA fine-tuned model using the same evaluation benchmark and evaluation protocol.

# 1. Environment Setup - Install Required Libraries

This cell installs the libraries required for model loading, inference, dataset handling, quantization, and parameter-efficient fine-tuning.

We use:

- **Transformers** - load and run Llama 3.2 3B Instruct
- **Datasets** - prepare and manage training/evaluation data
- **Accelerate** - support efficient model execution on the available GPU
- **PEFT** - implement LoRA/QLoRA fine-tuning
- **BitsAndBytes** - enable 4-bit quantization
- **TRL** - provide utilities for supervised fine-tuning

The goal at this stage is only to prepare the environment. No model weights are modified or fine-tuned in this cell.

# 1.1 Install required libraries

In [28]:
# cell 2
!pip install -q -U transformers datasets accelerate peft bitsandbytes trl

# 1.2 Verify environment and GPU
Environment and Hardware Verification

Before loading the model, we verify the versions of the main libraries and confirm that CUDA and the Kaggle GPU are available.

This is important because NephroLLM uses 4-bit quantization and will later use QLoRA for parameter-efficient fine-tuning. Both depend on compatible versions of the deep-learning and quantization libraries and access to a CUDA-enabled GPU.

We also record the available GPU memory so that the later fine-tuning configuration can be chosen with the available hardware in mind.

**Expected outcome:** CUDA should be available and a GPU should be detected.

In [29]:
# cell 4
import torch
import transformers
import datasets
import peft
import bitsandbytes

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )

PyTorch: 2.10.0+cu128
Transformers: 5.0.0
Datasets: 5.0.0
PEFT: 0.19.1
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


# 1.3 Authenticate with Hugging Face

Llama 3.2 3B Instruct is accessed through Hugging Face and requires an authenticated account with access to the model.

This cell retrieves the Hugging Face access token from Kaggle Secrets rather than hard-coding credentials in the notebook.

We only verify that a token is available. The token itself is never printed.

**Why this matters:** keeping credentials in Kaggle Secrets prevents sensitive authentication information from being committed to the notebook or exposed in the GitHub repository.

In [30]:
# cell 6
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

hf_token = user_secrets.get_secret("HF_TOKEN")

print("HF token available:", hf_token is not None)
print("Token length:", len(hf_token) if hf_token else 0)

HF token available: True
Token length: 37


# 2 - Load Llama 3.2 3B Instruct

### Loading the Base Model with 4-bit Quantization

We now load the pretrained **Llama 3.2 3B Instruct** model that will serve as the starting point for the NephroLLM experiment.

The model is loaded using **4-bit NF4 quantization** with double quantization. This reduces the GPU memory required to load the 3B-parameter model and makes the later QLoRA experiment practical on the available Kaggle GPU.

At this stage, the model is still completely **untuned**. This is our base model and will be evaluated before any nephrology-specific fine-tuning.

**Configuration**

- **Base model:** `meta-llama/Llama-3.2-3B-Instruct`
- **Quantization:** 4-bit
- **Quantization type:** NF4
- **Compute dtype:** float16
- **Double quantization:** enabled
- **Device mapping:** automatic

**Why 4-bit quantization?**

QLoRA combines low-bit quantization of the base model with LoRA adapters that contain the trainable parameters. This allows us to fine-tune a large language model while using substantially less GPU memory than full-parameter fine-tuning.

**Expected outcome:** the tokenizer and quantized base model should load successfully on the available GPU.

In [31]:
# cell 8 
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from kaggle_secrets import UserSecretsClient

# Get Hugging Face token securely
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

model_id = "meta-llama/Llama-3.2-3B-Instruct"

# 4-bit quantization configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    token=hf_token
)

print("Loading model...")

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    token=hf_token
)

print("Model loaded successfully!")
print("Model device:", model.device)

Loading tokenizer...
Loading model...


ImportError: Using `bitsandbytes` 4-bit quantization requires bitsandbytes: `pip install -U bitsandbytes>=0.46.1`

# Experimental Design

### Base Model → Fine-Tuning → Re-evaluation

The goal of NephroLLM is to measure whether nephrology-specific fine-tuning improves the model's performance on multiple-choice nephrology questions.

The experiment follows a controlled evaluation design:

1. **Base model evaluation**  
   Run the untuned Llama 3.2 3B Instruct model on the **636-question primary evaluation benchmark**.

2. **Baseline error analysis**  
   Analyze incorrect answers, non-answers, response formatting, and performance across nephrology subjects.

3. **Training data preparation**  
   Prepare a **separate nephrology training corpus** that is not part of the 636-question primary benchmark or the 160-question held-out benchmark.

4. **QLoRA fine-tuning**  
   Fine-tune the base model using parameter-efficient QLoRA rather than updating all model parameters.

5. **Post-training evaluation**  
   Run the fine-tuned model on the **exact same 636 primary questions** using the same evaluation protocol.

6. **Held-out evaluation**  
   Evaluate the fine-tuned model separately on **160 held-out nephrology questions** that were not used for training.

7. **Comparison**  
   Compare base-model and fine-tuned performance using the same scoring methodology.

### Evaluation Split

| Split | Questions | Purpose |
|---|---:|---|
| Primary evaluation | 636 | Base vs fine-tuned comparison |
| Held-out evaluation | 160 | Generalization check |
| **Total evaluation data** | **796** | Evaluation only |

**Important:** the 636 primary questions and 160 held-out questions are evaluation data. They must not be used as QLoRA training examples.

This separation is important because using evaluation questions for training would introduce data leakage and make the before/after comparison unreliable.

**Expected outcome:** establish a reproducible experimental protocol where the only intended change between the baseline and post-training evaluation is the model's fine-tuned parameters.

# 3 - Define & Verify the base Model Inference 

# 3.1 Define standardized inference

Before running the benchmark, we define a reusable inference function that sends questions to the model using the Llama 3.2 Instruct chat format.

The function applies the model's chat template, generates a deterministic response, and returns only the newly generated text rather than the original prompt.

### Generation protocol

- **Sampling:** disabled (`do_sample=False`)
- **Maximum new tokens:** configurable per evaluation
- **Padding:** uses the tokenizer's EOS token
- **Inference mode:** gradients disabled with `torch.no_grad()`

Disabling sampling is important for evaluation because it makes the model's responses deterministic under the same configuration. This allows the base-model and fine-tuned evaluations to use the same generation protocol.

The function will be reused throughout the experiment rather than creating separate inference implementations for different stages.

**Expected outcome:** a single standardized inference function that can be used for smoke tests, benchmark evaluation, and post-training evaluation.

In [ ]:
# cell 11
def ask_model(question, max_new_tokens=256):
    messages = [
        {
            "role": "system",
            "content": (
                "You are an educational assistant specializing in nephrology. "
                "Provide clear, factual, educational explanations. "
                "Do not provide diagnosis or personalized medical advice."
            )
        },
        {
            "role": "user",
            "content": question
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return answer.strip()

# 3.2 Run an inference smoke test

Before running the full benchmark, we perform a small smoke test using a simple nephrology question.

The purpose of this test is **not to measure model performance**. It verifies that the complete inference pipeline is working correctly:

- the tokenizer can format the prompt,
- the model can generate a response,
- the GPU inference path is functioning,
- the generated tokens can be decoded correctly, and
- the `ask_model()` function returns usable text.

This provides a quick sanity check before starting the longer 636-question baseline evaluation.

**Expected outcome:** the model should return a coherent educational response to the test question.

In [ ]:
# cell 13
question = "What is the primary function of the glomerulus?"

answer = ask_model(question)

print("QUESTION:")
print(question)

print("\nMODEL ANSWER:")
print(answer)

# 4. Load and validate the Nephrology Benchmark

# 4.1 Experimental and data strategy


### Separating Training Data from Evaluation Data

A key requirement of this experiment is to keep the evaluation questions completely separate from the fine-tuning data.

The NephroLLM benchmark currently provides two evaluation splits:

- **Primary evaluation:** 636 questions
- **Held-out evaluation:** 160 questions

There is **no training split in this benchmark**. Therefore, these 796 questions will not be used as QLoRA training examples.

### Role of Each Dataset

| Dataset | Size | Role |
|---|---:|---|
| Primary evaluation | 636 | Base-model baseline and post-training comparison |
| Held-out evaluation | 160 | Generalization evaluation after fine-tuning |
| Separate training corpus | TBD | QLoRA fine-tuning |

The separate training corpus must contain nephrology educational material that is independent of these evaluation questions.

### Why this separation matters

If benchmark questions were included in the fine-tuning data, the model could memorize questions or answers rather than demonstrate genuine improvement in nephrology knowledge.

Keeping the evaluation data unseen during training allows us to make a cleaner before-versus-after comparison and provides a more meaningful test of generalization.

**Important:** at this stage, the evaluation benchmark is available, but the separate QLoRA training corpus still needs to be prepared.

**Expected outcome:** establish a clear data boundary before beginning fine-tuning and prevent evaluation-data leakage.

# 4.2 Load the primary evaluation benchmark

### Evaluating the Base Model on the Primary Benchmark

We now evaluate the untuned Llama 3.2 3B Instruct model on the **636-question primary nephrology benchmark**.

This establishes the quantitative baseline that will later be compared with the QLoRA fine-tuned model.

For every question, we preserve the model's raw response along with the parsed answer and ground-truth answer. This allows us to distinguish between:

- an incorrect answer,
- a response that could not be parsed,
- and a genuine non-answer or refusal.

### Evaluation protocol

The same protocol will be reused after fine-tuning:

- **Questions:** 636 primary evaluation questions
- **Model:** Llama 3.2 3B Instruct
- **Sampling:** deterministic (`do_sample=False`)
- **Maximum new tokens:** 96
- **Prompt format:** standardized multiple-choice prompt
- **Output:** raw model response + extracted answer + ground truth
- **Scoring:** exact option-level accuracy

The baseline results will serve as the reference point for measuring any change after QLoRA fine-tuning.

**Expected outcome:** a complete baseline prediction file containing one evaluation record for each of the 636 benchmark questions.

In [ ]:
# cell 16
import os
import pandas as pd

# Find the finalized primary evaluation file
matches = []

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file == "nephrollm_primary_eval.csv":
            matches.append(os.path.join(root, file))

print("Found:", matches)

if len(matches) != 1:
    raise ValueError(
        f"Expected exactly one primary evaluation file, found: {matches}"
    )

eval_path = matches[0]
eval_df = pd.read_csv(eval_path)

print("Evaluation file:", eval_path)
print("Questions loaded:", len(eval_df))

display(
    eval_df[
        ["BenchmarkID", "Subject", "Question", "Choices", "Answer"]
    ].head()
)

# 5. Run a Fresh Baseline Evaluation

# 5.1 Lock the evaluation protocol
Locking the Evaluation Configuration

Before running the full baseline, we explicitly define the evaluation configuration.

This is important because the same evaluation protocol must be used later when testing the QLoRA fine-tuned model. Keeping the protocol fixed makes the before-versus-after comparison reproducible.

For each question, we will preserve:

1. The original benchmark question
2. The available answer choices
3. The model's raw response
4. The answer extracted by the evaluator
5. The ground-truth answer
6. Whether the extracted answer is correct

### Fixed evaluation settings

| Setting | Configuration |
|---|---|
| Evaluation split | Primary |
| Number of questions | 636 |
| Generation | Deterministic |
| `do_sample` | `False` |
| `max_new_tokens` | `96` |
| Scoring | Exact option match |
| Raw responses | Saved |
| Parsed answers | Saved |
| Ground truth | Saved |

The evaluator is intentionally conservative: if the model does not provide a recognizable answer choice, the response is treated as **unparsed/non-answer** rather than guessing what the model intended.

**Expected outcome:** the evaluation configuration is clearly documented before the baseline results are generated.

# 5.2 Define the MCQ answer parser

Multiple-Choice Answer Evaluator

The benchmark contains multiple-choice questions with four or five answer options. The model may express its answer in different formats, such as:

- `ANSWER: A`
- `ANSWER: 2`
- `C`
- `Option D`
- `Choice: B`

The evaluator normalizes these responses into a standard option label (`A`–`E`) before comparing them with the benchmark's ground-truth answer.

### Conservative parsing

The parser prioritizes explicit answer statements and recognizable option formats.

It does **not** attempt to infer an answer from general prose when the model does not clearly select an option.

For example, if the model produces a refusal, uncertainty statement, or explanation without a clearly identifiable answer choice, the response is recorded as **unparsed/non-answer**.

This conservative approach is intentional. Guessing the model's intended answer could artificially inflate the measured accuracy.

### Scoring

For each question:

```text
Predicted option == Ground-truth option
            ↓
        Correct / Incorrect

In [ ]:
import re

def extract_answer(response):
    """
    Extract a single answer choice.

    Supports alphabetical and numerical answer formats.
    Numerical choices are mapped by position:
    1=A, 2=B, 3=C, 4=D, 5=E
    """

    if not response:
        return None

    text = str(response).strip().upper()

    # 1. Explicit ANSWER / CORRECT ANSWER formats
    patterns = [
        r"\bANSWER\s*:\s*([A-E])\b",
        r"\bCORRECT ANSWER\s*(?:IS)?\s*:?\s*([A-E])\b",
        r"\bTHE ANSWER\s*(?:IS)?\s*:?\s*([A-E])\b",
        r"\bMY ANSWER\s*(?:IS)?\s*:?\s*([A-E])\b",
    ]

    for pattern in patterns:
        match = re.search(pattern, text)
        if match:
            return match.group(1)

    # 2. Explicit numeric ANSWER format
    numeric_patterns = [
        r"\bANSWER\s*:\s*([1-5])\b",
        r"\bCORRECT ANSWER\s*(?:IS)?\s*:?\s*([1-5])\b",
        r"\bTHE ANSWER\s*(?:IS)?\s*:?\s*([1-5])\b",
        r"\bMY ANSWER\s*(?:IS)?\s*:?\s*([1-5])\b",
    ]

    for pattern in numeric_patterns:
        match = re.search(pattern, text)
        if match:
            return "ABCDE"[int(match.group(1)) - 1]

    # 3. Standalone answer formats: A, A., A)
    lines = [x.strip() for x in text.splitlines() if x.strip()]

    for line in lines[:8]:
        match = re.fullmatch(r"([A-E])[\.\)]?", line)
        if match:
            return match.group(1)

    # 4. Standalone numeric answer: 1, 1., 1)
    for line in lines[:8]:
        match = re.fullmatch(r"([1-5])[\.\)]?", line)
        if match:
            return "ABCDE"[int(match.group(1)) - 1]

    # 5. OPTION / CHOICE format
    for line in lines[:8]:
        match = re.fullmatch(
            r"(?:OPTION|CHOICE)\s*[:\-]?\s*([A-E])[\.\)]?",
            line
        )
        if match:
            return match.group(1)

        match = re.fullmatch(
            r"(?:OPTION|CHOICE)\s*[:\-]?\s*([1-5])[\.\)]?",
            line
        )
        if match:
            return "ABCDE"[int(match.group(1)) - 1]

    return None

# 5.3 Define the standardized MCQ prompt

We now define the prompt used to evaluate each benchmark question.

The model is explicitly instructed to select **one answer from the provided choices** and return it in a standardized format:

```text
ANSWER: <option letter>

In [ ]:
# cell 19
def ask_mcq_fast(question, choices, max_new_tokens=32):
    """
    Ask the base model to answer one MCQ.

    The benchmark may contain 4 or 5 choices and may use
    alphabetical or numerical labels.
    """

    choice_lines = str(choices).strip()

    prompt = f"""Answer this multiple-choice nephrology question.

Question:
{question}

Choices:
{choice_lines}

Select the single best answer from the choices provided.

Return ONLY:
ANSWER: <option letter>
"""

    return ask_model(
        prompt,
        max_new_tokens=max_new_tokens
    )

# 5.4 Run a small evaluation sanity check


In [ ]:
# cell 27 
test_df = eval_df.head(5).copy()

test_results = []

for _, row in test_df.iterrows():

    response = ask_mcq_fast(
        row["Question"],
        row["Choices"]
    )

    predicted = extract_answer(response)
    expected = str(row["Answer"]).strip().upper()

    test_results.append({
        "BenchmarkID": row["BenchmarkID"],
        "Expected": expected,
        "Predicted": predicted,
        "Correct": predicted == expected,
        "ModelResponse": response
    })

test_results_df = pd.DataFrame(test_results)

display(test_results_df)

# 5.5 Run the full 636-question baseline

### Running the 636-Question Benchmark

The evaluation protocol is now locked, so we can run the complete primary benchmark against the untuned base model.

The benchmark contains **636 questions**. Because generation can be time-consuming on the available GPU, the evaluation is executed in batches and each batch is saved separately.

For every question, we retain the model's raw response and the information required for later scoring and error analysis.

### Why save batches?

Saving intermediate results provides two benefits:

1. **Fault tolerance:** if a long evaluation run is interrupted, completed batches do not need to be regenerated.
2. **Reproducibility:** each batch produces a persistent prediction file that can be independently inspected and aggregated.

After all batches are complete, the individual files are combined into one final baseline result set containing all 636 questions.

**Important:** this run is the base-model baseline. No QLoRA training has been performed yet.

**Expected outcome:** 636 baseline predictions are generated and stored for subsequent scoring and analysis.

In [ ]:
from tqdm.auto import tqdm
import os
import pandas as pd

BATCH_SIZE = 50

BASELINE_PREFIX = "/kaggle/working/nephrollm_baseline_final_batch_"

for start in range(0, len(eval_df), BATCH_SIZE):

    end = min(start + BATCH_SIZE, len(eval_df))
    batch = eval_df.iloc[start:end].copy()

    batch_number = start // BATCH_SIZE + 1

    print(
        f"\nRunning Batch {batch_number}: "
        f"questions {start + 1} to {end}"
    )

    batch_results = []

    for _, row in tqdm(
        batch.iterrows(),
        total=len(batch),
        desc=f"Batch {batch_number}"
    ):

        response = ask_mcq_fast(
            row["Question"],
            row["Choices"]
        )

        predicted = extract_answer(response)
        expected = str(row["Answer"]).strip().upper()

        batch_results.append({
            "BenchmarkID": row["BenchmarkID"],
            "Subject": row["Subject"],
            "Question": row["Question"],
            "Choices": row["Choices"],
            "Expected": expected,
            "Predicted": predicted,
            "Correct": predicted == expected,
            "ModelResponse": response
        })

    batch_df = pd.DataFrame(batch_results)

    batch_path = (
        f"{BASELINE_PREFIX}"
        f"{start + 1:03d}_{end:03d}.csv"
    )

    batch_df.to_csv(batch_path, index=False)

    print(f"Saved: {batch_path}")
    print(
        f"Batch accuracy: "
        f"{batch_df['Correct'].mean():.2%}"
    )
    print(
        f"Unparsed: "
        f"{batch_df['Predicted'].isna().sum()}"
    )

# 5.6 Aggregate baseline predictions

The 636-question baseline was executed in multiple batches. We now combine those batch outputs into a single evaluation dataset.

Before calculating the final score, we verify that:

- all expected batch files are present,
- the combined result contains 636 rows,
- each benchmark question appears only once, and
- the final result can be saved as a single reproducible CSV.

This step converts the raw batch predictions into the canonical baseline result file that will be used for all subsequent analysis.

**Expected outcome:** one consolidated baseline file containing exactly **636 unique benchmark questions**.

In [ ]:
import glob
import pandas as pd

baseline_files = sorted(
    glob.glob(
        "/kaggle/working/nephrollm_baseline_final_batch_*.csv"
    )
)

print("Fresh baseline batch files:", len(baseline_files))

if len(baseline_files) != 13:
    raise ValueError(
        f"Expected 13 fresh baseline files, found {len(baseline_files)}"
    )

baseline_df = pd.concat(
    [pd.read_csv(f) for f in baseline_files],
    ignore_index=True
)

# Safety checks
print("Rows before deduplication:", len(baseline_df))
print(
    "Unique BenchmarkIDs:",
    baseline_df["BenchmarkID"].nunique()
)

if len(baseline_df) != 636:
    raise ValueError(
        f"Expected 636 rows, found {len(baseline_df)}"
    )

if baseline_df["BenchmarkID"].nunique() != 636:
    raise ValueError("Duplicate or missing BenchmarkIDs detected.")

# Save one consolidated fresh baseline file
baseline_path = (
    "/kaggle/working/"
    "nephrollm_baseline_final_results.csv"
)

baseline_df.to_csv(
    baseline_path,
    index=False
)

print("\nFresh baseline saved:")
print(baseline_path)

print("\nColumns:")
print(list(baseline_df.columns))

# 6. Analyze Baseline Failures
### Measuring Base-Model Performance

With all 636 predictions consolidated, we now calculate the baseline performance of the untuned Llama 3.2 3B Instruct model.

We report two complementary metrics:

- **Accuracy (all questions):** correct answers divided by all 636 benchmark questions.
- **Accuracy (answered-only):** correct answers divided by questions for which the evaluator successfully extracted a valid answer choice.

We also report the number of **unparsed/non-answer responses** separately. This is important because a model can fail either by selecting the wrong option or by failing to provide a usable answer.

### Fresh baseline result

| Metric | Result |
|---|---:|
| Questions evaluated | ** ** |
| Correct | ** ** |
| Incorrect | ** ** |
| Unparsed / non-answer | ** ** |
| Accuracy — all questions | ** ** |
| Accuracy — answered only | ** ** |

The **41.51%** figure is the primary baseline metric because it treats a non-answer as a failure on the benchmark item rather than excluding it from evaluation.

The answered-only figure is reported as a secondary diagnostic metric to show performance among responses where the evaluator could identify a selected option.

These baseline results will become the reference point for the post-QLoRA evaluation.

**Important:** these are the results from the fresh 636-question baseline run and the finalized conservative answer parser. No fine-tuning has been performed yet.

# 6.1 Calculate the initial baseline results


In [ ]:
print("NEPHROLLM — FRESH BASELINE")
print("==========================")

total_questions = len(baseline_df)
correct = baseline_df["Correct"].sum()
unparsed = baseline_df["Predicted"].isna().sum()
incorrect = total_questions - correct - unparsed

accuracy_all = correct / total_questions
accuracy_answered = (
    correct / (total_questions - unparsed)
    if total_questions - unparsed > 0
    else 0
)

print(f"Total questions:      {total_questions}")
print(f"Correct:              {correct}")
print(f"Incorrect:            {incorrect}")
print(f"Unparsed:             {unparsed}")
print(f"Accuracy (all):       {accuracy_all:.2%}")
print(f"Accuracy (answered):  {accuracy_answered:.2%}")

# 6.2 Inspect unparsed responses

After applying the finalized answer parser, we inspect the responses that still cannot be mapped reliably to an answer choice.

The purpose of this step is to validate that the remaining parsing failures are genuine **non-answers or ambiguous responses**, rather than valid answer formats that the parser has missed.

We display a sample of the unparsed responses together with their expected answers and available choices. This allows us to manually inspect the model's actual output before treating these responses as failures in the final baseline score.

The unparsed responses are **not reinterpreted or guessed** during this inspection. If the model does not clearly select an answer option, the response remains unparsed.

**Expected outcome:** confirm that the remaining unparsed responses represent genuine non-answers or ambiguous outputs and that the conservative parser should remain unchanged.

In [ ]:
unparsed_df = baseline_df[
    baseline_df["Predicted"].isna()
].copy()

print("Unparsed responses:", len(unparsed_df))

display(
    unparsed_df[
        [
            "BenchmarkID",
            "Expected",
            "Choices",
            "ModelResponse"
        ]
    ].head(20)
)

# Re-scoring the Complete Baseline

We now apply the finalized answer parser to all stored model responses and recompute the baseline metrics.

The scoring logic separates three outcomes:

- **Correct:** the parsed model answer matches the benchmark answer.
- **Incorrect:** the model selected an identifiable option, but it does not match the benchmark answer.
- **Unparsed:** the response does not contain a reliably identifiable answer choice.

Two accuracy measures are calculated:

- **Accuracy (all):** correct answers divided by all 636 questions.
- **Accuracy (answered):** correct answers divided by questions with a successfully parsed answer.

The re-scored results are saved to:

`nephrollm_baseline_final_results.csv`

This is a deterministic post-processing step. **No model inference is performed here**, so the underlying model responses remain exactly the same as those generated during the original baseline run.

**Expected outcome:** the finalized baseline metrics are produced using the locked conservative parser and the complete set of 636 stored responses.

In [ ]:
baseline_df["Predicted"] = (
    baseline_df["ModelResponse"]
    .apply(extract_answer)
)

baseline_df["Correct"] = (
    baseline_df["Predicted"]
    == baseline_df["Expected"]
)

total_questions = len(baseline_df)
correct = baseline_df["Correct"].sum()
unparsed = baseline_df["Predicted"].isna().sum()
incorrect = total_questions - correct - unparsed

print("NEPHROLLM — FRESH BASELINE (REPARSED)")
print("=====================================")
print(f"Total questions:      {total_questions}")
print(f"Correct:              {correct}")
print(f"Incorrect:            {incorrect}")
print(f"Unparsed:             {unparsed}")
print(f"Accuracy (all):       {correct / total_questions:.2%}")
print(
    f"Accuracy (answered):  "
    f"{correct / (total_questions - unparsed):.2%}"
)

baseline_df.to_csv(
    "/kaggle/working/nephrollm_baseline_final_results.csv",
    index=False
)

# Final Inspection of Unparsed Responses

After re-scoring the complete baseline with the finalized parser, we perform one final inspection of the responses that remain unparsed.

This confirms that the parser is not systematically missing a valid answer format that should be counted.

For each remaining unparsed response, we retain:

- `BenchmarkID`
- expected answer
- available choices
- raw model response

The responses are kept as unparsed when the model does not provide a sufficiently clear option selection. We do not manually assign an answer during scoring.

This preserves a conservative and reproducible evaluation protocol.

**Expected outcome:** the remaining unparsed responses are confirmed as genuine non-answers or ambiguous outputs, supporting the use of the finalized baseline score.

In [ ]:
remaining_unparsed = baseline_df[
    baseline_df["Predicted"].isna()
].copy()

print("Remaining unparsed:", len(remaining_unparsed))

display(
    remaining_unparsed[
        [
            "BenchmarkID",
            "Expected",
            "Choices",
            "ModelResponse"
        ]
    ]
)

# 6.5 Analyze performance by subject

### Performance Across Nephrology Subjects

Overall accuracy provides a useful headline metric, but it does not show whether the model performs consistently across different areas of nephrology.

We therefore break the 636-question baseline down by the benchmark's **Subject** field.

For each subject, we examine:

- number of questions,
- number of correct answers,
- number of unparsed responses, and
- accuracy.

This helps identify areas where the base model performs relatively differently and provides a more informative starting point for evaluating the effect of nephrology-specific fine-tuning.

### Important interpretation note

Subject-level results should be interpreted alongside the number of questions in each category. Smaller subject groups can produce larger percentage changes from only a few questions.

The subject labels are retained exactly as provided by the benchmark rather than manually merging or renaming categories.

**Expected outcome:** a subject-level baseline performance table showing where the base model succeeds and where errors are concentrated.

In [ ]:
subject_accuracy = (
    baseline_df
    .groupby("Subject")
    .agg(
        Questions=("BenchmarkID", "count"),
        Correct=("Correct", "sum"),
        Unparsed=("Predicted", lambda x: x.isna().sum())
    )
    .reset_index()
)

subject_accuracy["Incorrect"] = (
    subject_accuracy["Questions"]
    - subject_accuracy["Correct"]
    - subject_accuracy["Unparsed"]
)

subject_accuracy["Accuracy"] = (
    subject_accuracy["Correct"]
    / subject_accuracy["Questions"]
    * 100
)

subject_accuracy = subject_accuracy.sort_values(
    "Accuracy",
    ascending=True
)

display(subject_accuracy)


# 6.6 Analyze incorrect answer selections


In [ ]:
# Baseline error analysis — parseable incorrect answers only

baseline_errors = baseline_df[
    (baseline_df["Predicted"].notna()) &
    (baseline_df["Correct"] == False)
].copy()

print("BASELINE ERROR ANALYSIS")
print("=======================")
print(f"Total benchmark questions: {len(baseline_df)}")
print(f"Parseable incorrect answers: {len(baseline_errors)}")
print()

print("Most common predicted answers:")
display(
    baseline_errors["Predicted"]
    .value_counts()
    .rename_axis("Predicted")
    .reset_index(name="Count")
)

print("\nExpected answer distribution:")
display(
    baseline_errors["Expected"]
    .value_counts()
    .rename_axis("Expected")
    .reset_index(name="Count")
)

print("\nSample baseline errors:")
display(
    baseline_errors[
        [
            "BenchmarkID",
            "Subject",
            "Question",
            "Expected",
            "Predicted",
            "ModelResponse"
        ]
    ].head(20)
)

# 6.7 Analyze response formats


In [ ]:
# Analyze baseline response formats

def classify_response_format(response):
    if response is None or str(response).strip() == "":
        return "empty"

    text = str(response).strip().upper()

    if re.search(r"\bANSWER\s*:\s*[A-E]\b", text):
        return "ANSWER: letter"

    if re.search(r"\bANSWER\s*:\s*[1-5]\b", text):
        return "ANSWER: number"

    if re.fullmatch(r"[A-E][\.\)]?", text):
        return "standalone letter"

    if re.fullmatch(r"[1-5][\.\)]?", text):
        return "standalone number"

    if text.startswith("I CANNOT") or text.startswith("I CAN'T"):
        return "refusal"

    if "MEDICAL ADVICE" in text:
        return "medical-advice refusal"

    return "other"


baseline_df["ResponseFormat"] = (
    baseline_df["ModelResponse"]
    .apply(classify_response_format)
)

format_summary = (
    baseline_df
    .groupby("ResponseFormat")
    .agg(
        Questions=("BenchmarkID", "count"),
        Correct=("Correct", "sum")
    )
    .reset_index()
)

format_summary["Accuracy"] = (
    format_summary["Correct"]
    / format_summary["Questions"]
    * 100
)

format_summary = format_summary.sort_values(
    "Questions",
    ascending=False
)

display(format_summary)

# 6.8 Audit generated baseline artifacts

In [ ]:
import os
import glob

print("Kaggle input datasets:")
print("======================")

for root, dirs, files in os.walk("/kaggle/input"):
    level = root.replace("/kaggle/input", "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")

    for file in files[:20]:
        print(f"{indent}  {file}")

print("\nWorking directory CSV files:")
print("============================")

for path in sorted(glob.glob("/kaggle/working/*.csv")):
    print(path)

# 4.3 Load the benchmark manifest and held-out dataset

### Loading the Evaluation Manifest and Held-Out Split

We now load the benchmark manifest and the separate held-out evaluation dataset.

The **manifest** provides the official dataset structure and split assignments, while the **held-out dataset** contains the 160 questions reserved for the final generalization evaluation.

At this stage, we are only inspecting the data. No held-out questions are used for model training.

We verify:

- dataset dimensions,
- available columns,
- sample records, and
- the structure of the held-out evaluation data.

### Why this matters

The held-out split must remain separate from the QLoRA training corpus. Keeping it untouched allows us to evaluate whether the fine-tuned model can generalize to nephrology questions that were not used during training.

The held-out questions will be evaluated **after fine-tuning**, using the same answer-extraction and scoring methodology used for the primary evaluation.

**Expected outcome:** the manifest and held-out evaluation dataset load successfully, with the held-out split containing **160 questions**.

In [ ]:
import pandas as pd

BASE = "/kaggle/input/datasets/shantanuss/nejm-ai-nephrology-benchmark"

manifest_path = f"{BASE}/nephrollm_evaluation_manifest.csv"
heldout_path = f"{BASE}/nephrollm_heldout_eval.csv"

manifest_df = pd.read_csv(manifest_path)
heldout_df = pd.read_csv(heldout_path)

print("MANIFEST")
print("========")
print("Shape:", manifest_df.shape)
print("Columns:", list(manifest_df.columns))
display(manifest_df.head(20))

print("\nHELD-OUT EVALUATION")
print("===================")
print("Shape:", heldout_df.shape)
print("Columns:", list(heldout_df.columns))
display(heldout_df.head(10))

# 4.4 Inspect benchmark structure and data quality

We now perform a more detailed structural check of the benchmark manifest.

The inspection covers:

- overall dataset shape,
- column names,
- sample records,
- data types, and
- missing values.

This confirms that the manifest contains the fields required for evaluation and split management, and helps identify any data-quality issues before the evaluation workflow continues.

The manifest contains **796 records across 10 columns**.

We specifically check for missing values rather than silently filling them, because the evaluation data should be preserved in its original form.

**Expected outcome:** the manifest structure and data types are consistent with the expected benchmark schema, with any missing values explicitly reported for review.

# 4.5 Validate official evaluation splits

### Confirming the Benchmark Split Assignments

Before proceeding with model evaluation and fine-tuning, we verify the official split assignments provided by the benchmark manifest.

The benchmark contains two evaluation splits:

- **`primary_eval` — 636 questions**
- **`held_out` — 160 questions**

There is no training split in the benchmark manifest.

We also inspect the subject distribution within each split to understand how the evaluation data is organized.

### Why this validation matters

The split assignments determine how the benchmark will be used throughout the experiment:

- `primary_eval` is used for the base-model baseline and the later base-vs-fine-tuned comparison.
- `held_out` is reserved for the separate generalization evaluation.
- Neither split is permitted to become part of the QLoRA training corpus.

This check establishes the data boundary before any fine-tuning begins.

**Expected outcome:** the manifest should contain exactly **636 primary-evaluation questions and 160 held-out questions**, for a total of **796 evaluation questions**.

In [ ]:
# Compact manifest inspection

print("Manifest shape:", manifest_df.shape)
print("Manifest columns:", list(manifest_df.columns))

print("\nFirst 10 rows:")
display(manifest_df.head(10))

print("\nData types:")
display(manifest_df.dtypes.to_frame("dtype"))

print("\nMissing values:")
display(
    manifest_df.isna().sum()
    .sort_values(ascending=False)
    .to_frame("Missing")
)

In [ ]:
# Inspect the official benchmark split assignments

print("Official split distribution")
print("===========================")

split_counts = (
    manifest_df["split"]
    .value_counts()
    .rename_axis("Split")
    .reset_index(name="Questions")
)

display(split_counts)

print("\nSubject distribution by split")
print("=============================")

split_subject_counts = (
    manifest_df
    .groupby(["split", "Subject"])
    .size()
    .reset_index(name="Questions")
)

display(split_subject_counts)

# 7. Prepare fine-tuning dataset


# 7.1 Define training-data requirements

The benchmark used in this project is reserved entirely for evaluation. It contains 636 primary-evaluation questions and 160 held-out questions, with no training split.

Therefore, the QLoRA training corpus must be a **separate nephrology educational dataset** that is not derived from the evaluation questions.

The training corpus should provide examples that are suitable for supervised instruction fine-tuning, with a clear relationship between an input and an expected educational response.

### Training-data requirements

The training corpus should:

- contain nephrology-focused educational content;
- be independent of the 636 primary evaluation questions;
- be independent of the 160 held-out questions;
- provide sufficiently clear question/answer or instruction/response examples;
- avoid relying on patient-specific medical advice;
- be convertible into the chat/instruction format expected by Llama 3.2 3B Instruct.

### Data-leakage rule

The evaluation benchmark is treated as **read-only evaluation data**.

No question, answer, solution, or explanation from the primary or held-out evaluation sets will be intentionally included as a QLoRA training example.

This separation is necessary so that the later comparison between the base model and fine-tuned model measures performance on unseen evaluation data rather than memorization.

**Expected outcome:** establish the requirements and data boundaries for the separate QLoRA training corpus before any training data is loaded or transformed.

# 7.2 Select an independent nephrology theory corpus


The evaluation benchmark does not contain a training split, so the QLoRA training data must come from a separate source.

For NephroLLM, we will use an **independent medical theory/textbook corpus** as the knowledge source for fine-tuning rather than training directly on multiple-choice questions.

### Why a theory corpus?

The goal of fine-tuning is to improve the model's underlying nephrology knowledge and its ability to provide educational explanations.

A theory-based corpus can expose the model to:

- nephrology concepts and terminology;
- disease mechanisms;
- renal physiology and pathophysiology;
- diagnostic principles;
- treatment concepts; and
- explanatory relationships between clinical concepts.

This provides a broader knowledge signal than training exclusively on MCQs.

### Training-data strategy

The selected theory corpus will be processed through the following pipeline:

```text
Independent medical theory corpus
            ↓
Identify nephrology-relevant content
            ↓
Clean and validate the text
            ↓
Remove duplicates and unsuitable content
            ↓
Check for overlap with the NephroLLM benchmark
            ↓
Convert selected content into
instruction-response examples
            ↓
Final QLoRA training dataset

# 7.3 Load and Inspect the Theory Corpus

We now load the independent medical theory corpus selected as the knowledge source for NephroLLM fine-tuning.

The purpose of this step is to understand the raw corpus before extracting nephrology-specific content.

We will inspect:

- the dataset structure and available splits;
- the available metadata;
- the source documents or textbooks;
- the text fields;
- the approximate size of the corpus; and
- whether the corpus provides enough information to reliably identify nephrology-related content.

At this stage, we will **not** filter, transform, or generate training examples.

Understanding the raw data first allows us to design the subsequent filtering and preprocessing steps based on the actual corpus structure rather than making assumptions about its fields or organization.

### Data boundary

The theory corpus is an independent training source.

The **636 primary-evaluation questions** and **160 held-out questions** remain outside this pipeline and will not be used to construct training examples.

**Expected outcome:** confirm that the theory corpus loads successfully and understand its schema and available metadata before extracting nephrology-relevant content.

In [ ]:
from datasets import load_dataset

# Load the MedRAG medical textbook corpus
theory_ds = load_dataset("MedRAG/textbooks")

# Basic dataset structure
print("Dataset:")
print(theory_ds)

# Inspect available splits
print("\nSplits:")
print(theory_ds.keys())

# Inspect schema
print("\nFeatures:")
print(theory_ds["train"].features)

# Number of records
print("\nNumber of records:")
print(len(theory_ds["train"]))

# Inspect a sample record
print("\nSample record:")
print(theory_ds["train"][0])

# 7.4 Identify Nephrology-Relevant Content

The theory corpus contains medical content across multiple subjects and clinical domains. We now identify the portions that are relevant to nephrology and renal medicine.

The goal is to construct a focused nephrology knowledge corpus rather than fine-tuning on unrelated medical material.

### Selection approach

We will use the metadata and text available in the corpus to identify relevant content.

Potential nephrology-related concepts include:

- kidney and renal physiology;
- acute kidney injury;
- chronic kidney disease;
- glomerular diseases;
- electrolyte and acid-base disorders;
- hypertension and kidney disease;
- dialysis and end-stage kidney disease;
- renal transplantation;
- nephrolithiasis;
- renal pathology; and
- other kidney-related conditions and mechanisms.

The exact filtering criteria will be determined from the actual structure and metadata of the loaded corpus.

### Important consideration

Keyword matching alone can produce both false positives and false negatives.

For example, a general medical passage may mention the kidney without being primarily about nephrology, while a relevant nephrology concept may not contain an obvious keyword.

Therefore, the initial filter will be treated as a **candidate-selection step**. The resulting records will be inspected before they become part of the final training corpus.

**Expected outcome:** identify a reproducible method for extracting candidate nephrology-related passages from the independent theory corpus.

In [ ]:
import pandas as pd
import re

# Convert the training split to a DataFrame
theory_df = theory_ds["train"].to_pandas()

# Textbooks most likely to contain substantial nephrology / renal content
relevant_books = [
    "InternalMed_Harrison",
    "Physiology_Levy",
    "Pathology_Robbins",
    "Pathoma_Husain",
    "Anatomy_Gray",
    "Pharmacology_Katzung",
]

# Nephrology / renal keywords used for candidate selection
nephro_keywords = [
    "nephrology",
    "nephrolog",
    "renal",
    "kidney",
    "glomerul",
    "nephron",
    "glomerulonephritis",
    "nephritis",
    "nephrotic",
    "glomerulosclerosis",
    "acute kidney injury",
    "acute renal failure",
    "chronic kidney disease",
    "chronic renal failure",
    "dialysis",
    "hemodialysis",
    "peritoneal dialysis",
    "renal replacement",
    "kidney transplant",
    "renal transplant",
    "renal transplantation",
    "electrolyte",
    "acid-base",
    "hyperkalemia",
    "hypokalemia",
    "hyponatremia",
    "hypernatremia",
    "renal stone",
    "kidney stone",
    "nephrolithiasis",
    "polycystic kidney",
    "renal artery",
    "renal tubular",
    "tubulointerstitial",
]

# Restrict to selected medical textbooks
candidate_df = theory_df[
    theory_df["title"].isin(relevant_books)
].copy()

# Normalize text for case-insensitive matching
candidate_df["content_lower"] = (
    candidate_df["content"]
    .fillna("")
    .str.lower()
)

# Escape keywords before constructing the regex
keyword_pattern = "|".join(
    map(re.escape, nephro_keywords)
)

# Identify candidate nephrology chunks
candidate_df = candidate_df[
    candidate_df["content_lower"].str.contains(
        keyword_pattern,
        regex=True,
        na=False
    )
].copy()

# Remove temporary column
candidate_df.drop(columns=["content_lower"], inplace=True)

print("Candidate nephrology chunks:", len(candidate_df))

print("\nCandidate chunks by textbook:")
print(
    candidate_df["title"]
    .value_counts()
    .sort_index()
)

Inspect Candidate Nephrology Content

In [ ]:
# Inspect representative candidate chunks from each source textbook

sample_per_book = 3

for book in relevant_books:
    book_samples = candidate_df[
        candidate_df["title"] == book
    ].sample(
        n=min(sample_per_book, len(candidate_df[candidate_df["title"] == book])),
        random_state=42
    )

    print("\n" + "=" * 100)
    print(f"SOURCE: {book}")
    print("=" * 100)

    for i, row in book_samples.iterrows():
        print(f"\nID: {row['id']}")
        print(f"TEXT:\n{row['content'][:1500]}")
        print("-" * 100)

In [ ]:
# Stronger nephrology-specific terms for second-stage filtering

strong_nephro_keywords = [
    "glomerul",
    "nephron",
    "nephritis",
    "nephrotic",
    "nephropathy",
    "nephrolithiasis",
    "glomerulosclerosis",
    "glomerulonephritis",
    "acute kidney injury",
    "acute renal failure",
    "chronic kidney disease",
    "chronic renal failure",
    "end-stage renal",
    "end stage renal",
    "renal replacement therapy",
    "renal replacement",
    "hemodialysis",
    "haemodialysis",
    "peritoneal dialysis",
    "dialysis",
    "kidney transplant",
    "renal transplant",
    "renal transplantation",
    "renal tubular",
    "tubular acidosis",
    "tubulointerstitial",
    "renal artery",
    "renal vein",
    "renal blood flow",
    "glomerular filtration",
    "glomerular filtration rate",
    "glomerular capillary",
    "glomerular disease",
    "renal stone",
    "kidney stone",
    "renal calcul",
    "polycystic kidney",
    "renal failure",
    "renal insufficiency",
    "renal disease",
    "renal physiology",
    "renal pathology",
]

strong_pattern = "|".join(
    map(re.escape, strong_nephro_keywords)
)

# Apply the stronger filter to the existing candidate pool
candidate_df["content_lower"] = (
    candidate_df["content"]
    .fillna("")
    .str.lower()
)

nephro_df = candidate_df[
    candidate_df["content_lower"].str.contains(
        strong_pattern,
        regex=True,
        na=False
    )
].copy()

nephro_df.drop(columns=["content_lower"], inplace=True)

print("Initial candidate chunks:", len(candidate_df))
print("Stronger nephrology candidates:", len(nephro_df))

print("\nCandidates by textbook:")
print(
    nephro_df["title"]
    .value_counts()
    .sort_index()
)

Validate the Refined Nephrology Candidates

In [ ]:
# Inspect representative examples from the refined nephrology candidate set

sample_per_book = 5

for book in relevant_books:
    book_df = nephro_df[nephro_df["title"] == book]

    if len(book_df) == 0:
        continue

    samples = book_df.sample(
        n=min(sample_per_book, len(book_df)),
        random_state=42
    )

    print("\n" + "=" * 100)
    print(f"SOURCE: {book} | CANDIDATES: {len(book_df)}")
    print("=" * 100)

    for _, row in samples.iterrows():
        print(f"\nID: {row['id']}")
        print(f"TEXT:\n{row['content'][:1800]}")
        print("-" * 100)

# 7.5 Clean and Prepare the Training Text

After identifying candidate nephrology content, we now prepare the selected text for use in supervised fine-tuning.

The objective is to retain useful medical knowledge while removing records that are unsuitable or likely to introduce noise into the training process.

### Cleaning steps

The preprocessing stage will examine the selected records for:

- missing or empty text;
- duplicate passages;
- extremely short or excessively long passages;
- malformed records;
- unnecessary formatting or metadata; and
- passages that do not contain meaningful educational content.

We will preserve the original educational meaning of the source material while applying only transformations necessary for training-data preparation.

### Why cleaning matters

Fine-tuning quality depends not only on the quantity of training data but also on the consistency and usefulness of the examples.

Removing duplicates and unusable records helps prevent the model from receiving repeated or low-information training examples.

At the same time, we avoid aggressive preprocessing that could remove medically meaningful terminology or context.

**Expected outcome:** a cleaned set of nephrology-focused educational passages suitable for conversion into instruction-response training examples.

In [ ]:
# Basic cleaning and quality checks for the nephrology candidate corpus

import re
import pandas as pd

clean_df = nephro_df.copy()

# Normalize whitespace while preserving the original medical content
clean_df["content_clean"] = (
    clean_df["content"]
    .fillna("")
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Basic text statistics
clean_df["char_count"] = clean_df["content_clean"].str.len()
clean_df["word_count"] = clean_df["content_clean"].str.split().str.len()

print("Initial nephrology candidates:", len(clean_df))

print("\nMissing / empty content:")
print(
    (clean_df["content_clean"].str.len() == 0).sum()
)

print("\nText length statistics:")
print(
    clean_df["word_count"].describe()
)

print("\nVery short passages (<50 words):")
print(
    (clean_df["word_count"] < 50).sum()
)

print("\nVery long passages (>1500 words):")
print(
    (clean_df["word_count"] > 1500).sum()
)

print("\nDuplicate passages:")
print(
    clean_df["content_clean"].duplicated().sum()
)

inspect short passages

In [ ]:
# Inspect short nephrology passages before deciding whether to exclude them

short_df = clean_df[
    clean_df["word_count"] < 50
].copy()

print("Short passages:", len(short_df))

print("\nShort passages by textbook:")
print(
    short_df["title"]
    .value_counts()
    .sort_index()
)

print("\nRepresentative short passages:")

for book in relevant_books:
    book_samples = short_df[
        short_df["title"] == book
    ]

    if len(book_samples) == 0:
        continue

    samples = book_samples.sample(
        n=min(5, len(book_samples)),
        random_state=42
    )

    print("\n" + "=" * 100)
    print(f"SOURCE: {book}")
    print("=" * 100)

    for _, row in samples.iterrows():
        print(f"\nID: {row['id']}")
        print(f"WORDS: {row['word_count']}")
        print(f"TEXT: {row['content_clean']}")
        print("-" * 100)

remove obvious structural noise and duplicates

We'll first remove only clearly unusable records:

empty text;
extremely tiny fragments (<10 words);
obvious figure/table fragments;
duplicate text.

We will keep passages of 10+ words for now, even if they're short, because some contain dense nephrology knowledge.

In [ ]:
# Apply conservative cleaning rules to the nephrology candidate corpus

before_count = len(clean_df)

# Remove empty passages
clean_df = clean_df[
    clean_df["content_clean"].str.len() > 0
].copy()

# Remove extremely short fragments
# We keep passages with >=10 words because some short passages
# contain dense nephrology concepts.
clean_df = clean_df[
    clean_df["word_count"] >= 10
].copy()

# Remove exact duplicate passages
clean_df = clean_df.drop_duplicates(
    subset=["content_clean"]
).copy()

# Reset index after filtering
clean_df.reset_index(drop=True, inplace=True)

after_count = len(clean_df)

print("Candidate passages before cleaning:", before_count)
print("Candidate passages after cleaning:", after_count)
print("Removed passages:", before_count - after_count)

print("\nRemaining passages by textbook:")
print(
    clean_df["title"]
    .value_counts()
    .sort_index()
)

print("\nRemaining word-count statistics:")
print(
    clean_df["word_count"].describe()
)

# 7.6 Convert Theory into Instruction-Response Examples

The cleaned nephrology passages are still raw educational text. We now transform them into structured **instruction-response examples** suitable for supervised fine-tuning.

The objective is to teach Llama 3.2 3B Instruct to use nephrology knowledge when responding to educational prompts rather than simply memorizing isolated text passages.

### Example structure

Each training example will follow a simple instruction-response format:

```text
Instruction:
Explain the role of the glomerulus in renal filtration.

Response:
The glomerulus is a network of specialized capillaries...

**Why are we converting to Instruction response format ?**

Because Llama 3.2 3B Instruct is an instruction-tuned model, and our goal is not simply to make it memorize textbook passages. We want QLoRA to improve its ability to answer nephrology questions and explain nephrology concepts.

There are two different training approaches:
| Approach                         | What the model sees                  | What it teaches                                                 |
| -------------------------------- | ------------------------------------ | --------------------------------------------------------------- |
| Raw text / continued pretraining | `"The glomerulus filters plasma..."` | More exposure to nephrology language/knowledge                  |
| Instruction-response SFT         | **Question → Answer**                | How to use nephrology knowledge to answer educational questions |


**Identify Passages Suitable for Instruction Generation**

Before generating instruction-response pairs, let's first identify passages that contain enough coherent nephrology information to support an educational question and answer.

For this step, we'll use simple heuristics, not an LLM. This keeps the selection reproducible and avoids introducing synthetic judgments too early.

In [ ]:
# Identify cleaned passages that are suitable candidates
# for instruction-response generation.

instruction_candidates = clean_df.copy()

# Basic suitability signals
instruction_candidates["has_explanatory_language"] = (
    instruction_candidates["content_clean"]
    .str.contains(
        r"\b(is|are|refers to|defined|causes|results in|leads to|"
        r"associated with|characterized by|regulates|inhibits|"
        r"stimulates|reabsorbs|excretes|increases|decreases)\b",
        case=False,
        regex=True,
        na=False
    )
)

instruction_candidates["has_nephro_term"] = (
    instruction_candidates["content_clean"]
    .str.contains(
        r"\b(kidney|renal|nephron|nephritic|nephrotic|"
        r"glomerul|dialysis|glomerular|tubular|"
        r"ureter|urine|creatinine|GFR|"
        r"aldosterone|renin|electrolyte|acid-base)\b",
        case=False,
        regex=True,
        na=False
    )
)

# Keep passages that contain enough text and at least
# one explanatory or nephrology-specific signal.
instruction_candidates = instruction_candidates[
    (instruction_candidates["word_count"] >= 20) &
    (
        instruction_candidates["has_explanatory_language"] |
        instruction_candidates["has_nephro_term"]
    )
].copy()

print("Cleaned passages:", len(clean_df))
print("Instruction-generation candidates:", len(instruction_candidates))
print(
    "Candidate retention:",
    f"{len(instruction_candidates) / len(clean_df) * 100:.1f}%"
)

print("\nCandidates by textbook:")
print(
    instruction_candidates["title"]
    .value_counts()
    .sort_index()
)

# Remove helper columns before continuing
instruction_candidates.drop(
    columns=[
        "has_explanatory_language",
        "has_nephro_term"
    ],
    inplace=True
)


3,024 of 3,113 passages (97.1%) pass the basic suitability heuristic. That tells us the heuristic is intentionally permissive—which is good here. We don't want to discard potentially useful medical content before we have actually transformed it.

**Define the Instruction-Response Format**

The selected theory passages now need to be transformed into supervised fine-tuning examples.

Rather than generating arbitrary questions, we will use a small set of controlled educational instruction types.

### Instruction types

Depending on the information available in a passage, an example may ask the model to:

- **Explain a concept** — explain a nephrology concept or mechanism.
- **Describe a process** — describe how a renal physiological or pathological process works.
- **Explain a clinical concept** — explain a disease, syndrome, finding, or complication.
- **Explain a relationship** — explain how two nephrology concepts are related.
- **Compare concepts** — compare two concepts when the source passage explicitly supports the comparison.

Not every passage needs to support every instruction type.

### Example

For a passage describing glomerular filtration:

```text
Instruction:
Explain the role of the glomerulus in renal filtration.

Response:
The glomerulus is a specialized network of capillaries that filters
plasma to form the initial tubular fluid...

**Create a Pilot Instruction-Response Dataset**

In [ ]:
# Create a small pilot dataset from the selected nephrology passages.
# The pilot will be inspected before generating the full training set.

pilot_size = min(50, len(instruction_candidates))

pilot_df = instruction_candidates.sample(
    n=pilot_size,
    random_state=42
).copy()

# Define the instruction-response dataset schema.
# The response will be generated in the next step.
pilot_df["instruction"] = ""
pilot_df["response"] = ""

# Keep source information for traceability.
pilot_training_df = pilot_df[
    [
        "id",
        "title",
        "content_clean",
        "instruction",
        "response"
    ]
].copy()

print("Pilot passages:", len(pilot_training_df))

print("\nDataset columns:")
print(pilot_training_df.columns.tolist())

print("\nSample source passages:")
for _, row in pilot_training_df.head(5).iterrows():
    print("\n" + "=" * 80)
    print(f"Source: {row['title']}")
    print(f"ID: {row['id']}")
    print(f"Text: {row['content_clean']}")

The pilot reveals exactly why we did this check. The passages are not all equally suitable for SFT generation.

For example:

The cortisol/kidney passage is coherent and can support several educational questions.
The dRTA passage is also rich enough for an educational example.
The BUN/creatinine passage is strong.
But the thyroid passage contains an incidental renal reference.
The Sjögren's passage is essentially a table/summary fragment, not a good standalone teaching passage.

So before generating synthetic instructions, **we should add one more quality-control layer.**

In [ ]:
# Identify passages that are likely to be structurally noisy,
# such as table fragments, figure labels, or broken text.

quality_df = instruction_candidates.copy()

text = quality_df["content_clean"]

# Signals of potentially noisy textbook extraction
quality_df["has_table_signal"] = text.str.contains(
    r"\b(Table|TABLE|Fig\.|Figure|FIGURE)\b",
    regex=True,
    na=False
)

quality_df["has_fragment_signal"] = (
    text.str.match(r"^[a-z]", na=False) |
    text.str.contains(r"\.{2,}|:{2,}", regex=True, na=False)
)

quality_df["has_many_short_tokens"] = (
    text.str.split().apply(
        lambda words: (
            len(words) > 0 and
            sum(len(w) <= 2 for w in words) / len(words) > 0.25
        )
    )
)

# Count passages with each signal
print("Instruction candidates:", len(quality_df))

print("\nPotential table/figure passages:")
print(quality_df["has_table_signal"].sum())

print("\nPotential fragmented passages:")
print(quality_df["has_fragment_signal"].sum())

print("\nPassages with many very short tokens:")
print(quality_df["has_many_short_tokens"].sum())

# Show examples flagged by the structural checks
flagged_df = quality_df[
    quality_df["has_table_signal"] |
    quality_df["has_fragment_signal"] |
    quality_df["has_many_short_tokens"]
].copy()

print("\nTotal flagged for review:", len(flagged_df))

for _, row in flagged_df.head(15).iterrows():
    print("\n" + "=" * 100)
    print(f"SOURCE: {row['title']}")
    print(f"ID: {row['id']}")
    print(f"TEXT:\n{row['content_clean']}")

This confirms the structural filter is too broad to use as an automatic exclusion rule.

We have 996 flagged passages, but several are actually useful. For example, a passage can contain a legitimate renal concept while also mentioning a figure/table. Conversely, some flagged passages are clearly incidental—for example, renal disease appearing only as one risk factor or comorbidity.

The purpose of the generation prompt should therefore be simply:

**Take an already-selected nephrology passage and create one instruction-response example grounded in that passage.**

In [ ]:
def create_training_example_prompt(passage):
    return f"""Convert the following nephrology educational passage
into ONE instruction-response training example.

Use ONLY information supported by the passage.

Return exactly:

INSTRUCTION: <one clear nephrology educational question>
RESPONSE: <a clear educational answer based on the passage>

Rules:
- Focus on the main nephrology concept in the passage.
- Do not introduce information not supported by the passage.
- Do not provide patient-specific medical advice.
- Do not create multiple questions.
- Prefer conceptual or explanatory questions.
- Keep the response concise but informative.

SOURCE PASSAGE:
{passage}
"""

**Generate 10 Pilot Instruction-Response Examples**

In [ ]:
# Generate a small pilot of instruction-response examples

import torch
import pandas as pd

pilot_results = []

# Use 10 passages from our existing 50-passage pilot
generation_pilot = pilot_training_df.head(10)

model.eval()

for _, row in generation_pilot.iterrows():

    prompt = create_training_example_prompt(
        row["content_clean"]
    )

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][
        inputs["input_ids"].shape[1]:
    ]

    generated_text = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    pilot_results.append({
        "source_id": row["id"],
        "source_title": row["title"],
        "source_text": row["content_clean"],
        "generated_example": generated_text
    })

# Store the results without modifying the original dataset
generated_pilot_df = pd.DataFrame(pilot_results)

print("Generated examples:", len(generated_pilot_df))

for _, row in generated_pilot_df.iterrows():
    print("\n" + "=" * 90)
    print("SOURCE ID:", row["source_id"])
    print("SOURCE:", row["source_title"])
    print("\nGENERATED EXAMPLE:")
    print(row["generated_example"])

# 7.7 Check for Evaluation Leakage

Before finalizing the training dataset, we perform an explicit leakage check against the NephroLLM evaluation benchmark.

The evaluation benchmark contains:

- **636 primary-evaluation questions**
- **160 held-out questions**

None of these evaluation records should be intentionally present in the QLoRA training data.

### Leakage checks

We will compare the candidate training examples against the evaluation benchmark using available identifying information and normalized text.

The check will look for:

- exact question matches;
- exact answer or explanation matches where applicable;
- duplicated or near-duplicated question text; and
- obvious overlap between source-derived training examples and evaluation items.

### Why this matters

A fine-tuned model could appear to improve simply because it has been exposed to evaluation questions or their answers during training.

That would make the before-versus-after comparison difficult to interpret.

The leakage check therefore acts as a quality-control step before the final training dataset is created.

### Evaluation boundary

```text
Training corpus
      │
      ├── Allowed → independent nephrology educational content
      │
      └── Excluded → evaluation benchmark content

Evaluation benchmark
      │
      ├── 636 primary questions
      └── 160 held-out questions

# 7.8 Prepare the Final QLoRA Dataset

After cleaning the nephrology content and completing the evaluation-leakage check, we now prepare the final dataset for supervised fine-tuning.

The dataset will contain structured instruction-response examples compatible with the training pipeline used in the QLoRA stage.

### Final training-example structure

Each record will contain:

- an instructional prompt;
- the corresponding educational response; and
- the source reference where available.

The examples will then be formatted using the conversational structure expected by **Llama 3.2 3B Instruct**.

### Final quality checks

Before training begins, we will verify:

- the dataset is not empty;
- all required fields are present;
- instruction and response fields contain usable text;
- duplicate examples have been removed;
- extremely short or malformed examples have been excluded;
- the examples remain focused on nephrology; and
- no evaluation benchmark questions have been intentionally included.

We will also record the final number of training examples so that the fine-tuning experiment is reproducible.

### Training boundary

At the end of this stage:

```text
Independent theory corpus
          ↓
Nephrology filtering
          ↓
Cleaning
          ↓
Instruction-response conversion
          ↓
Leakage check
          ↓
Final QLoRA training dataset
          ↓
Section 8: QLoRA fine-tuning

# 8. Fine-tune using QLoRA

# 8.1 Explain QLoRA configuration

# 8.2 Configure LoRA adapters


# 8.3 Configure training arguments


# 8.4 Train the model


# 8.5 Inspect training metrics

# 9. Save the LoRA adapter

# 9.1 Save adapter

# 9.2 Verify saved artifacts

# 9.3 Save tokenizer/configuration if required

# 10. Run the exact same evaluation on the fine-tuned model

# 10.1 Load the fine-tuned adapter

# 10.2 Reuse the same inference protocol

# 10.3 Reuse the same MCQ prompt

# 10.4 Reuse the same parser

# 10.5 Evaluate the same 636 questions


# 10.6 Aggregate results

# 11. Evaluate on held-out questions


# 11.1 Run inference on 160 held-out questions

# 11.2 Parse responses


# 11.3 Calculate held-out accuracy


# 11.4 Analyze unparsed responses

# 12. Compare base vs fine-tuned model


# 12.1 Overall accuracy comparison

# 12.2 Answered-only accuracy comparison


# 12.3 Unparsed-response comparison


# 12.4 Subject-level comparison


# 12.5 Error-pattern comparison


# 12.6 Held-out comparison

# 13. Document findings, limitations, and conclusions


# 13.1 Key findings


# 13.2 What improved


# 13.3 What did not improve

# 13.4 Limitations


# 13.5 Data leakage considerations


# 13.6 Conclusion 